# Structured Outputs

<br>

## What Are Structured Outputs?

LLMs normally return free-form text. That is useful for people, but applications often need predictable data such as a category, priority, or list of tags.

**Structured Outputs** make the model follow a schema that defines the expected fields and their types. The result can be used directly in Python without manually parsing prose.

This is stronger than simply asking for JSON: valid JSON can still contain missing fields, unexpected keys, or the wrong values.

<br>

## Initial Setup

To describe the output schema, we will use **Pydantic**, a popular Python library for data validation using type annotations.

The OpenAI Python SDK converts the model into JSON Schema and parses the response back into a validated Python object.

In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())  # read local .env file
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

<br>

## Step 1 — Define the Output Schema

Suppose a support team wants to turn customer messages into structured tickets.

Each field has a specific type. `Literal` restricts a field to a fixed set of allowed values, while `Field` descriptions help the model understand what to extract.

In [2]:
class SupportTicket(BaseModel):
    # Field adds extra information to a schema property.
    # The description tells the model what this value should contain.
    summary: str = Field(description="A short summary of the customer's problem")

    # Literal restricts a value to one of the listed options.
    # The model must choose exactly one of these categories.
    category: Literal["billing", "technical", "account", "other"]

    # Another closed set of allowed values: "low", "medium", or "high".
    priority: Literal["low", "medium", "high"]

    # list[str] means a list of strings.
    # Field's description guides what those labels should represent.
    tags: list[str] = Field(description="A short list of useful labels")


print(SupportTicket.model_json_schema())

{'properties': {'summary': {'description': "A short summary of the customer's problem", 'title': 'Summary', 'type': 'string'}, 'category': {'enum': ['billing', 'technical', 'account', 'other'], 'title': 'Category', 'type': 'string'}, 'priority': {'enum': ['low', 'medium', 'high'], 'title': 'Priority', 'type': 'string'}, 'tags': {'description': 'A short list of useful labels', 'items': {'type': 'string'}, 'title': 'Tags', 'type': 'array'}}, 'required': ['summary', 'category', 'priority', 'tags'], 'title': 'SupportTicket', 'type': 'object'}


<br>

## Step 2 — Ask the Model for Structured Data

Use `client.responses.parse()` and pass the Pydantic model as `text_format`. The SDK sends the schema to the model and validates the returned data.

In [15]:
message = """
I have been charged twice for my subscription this month.
Please fix this soon because I need the money back.
"""

response = client.responses.parse(
    model="gpt-5.4-nano",
    input=[
        {"role": "system", "content": "Extract the customer message into a support ticket."},
        {"role": "user", "content": message},
    ],
    text_format=SupportTicket,
)

ticket = response.output_parsed
print(ticket)


summary='Customer reports being charged twice for their subscription in the current month and requests a refund.' category='billing' priority='high' tags=['duplicate_charge', 'subscription', 'refund_request', 'billing_discrepancy']


<br>

## Step 3 — Use the Parsed Python Object

`response.output_parsed` is a `SupportTicket` object, not a JSON string. Its fields can be used directly in application code.

In [4]:
print("Summary :", ticket.summary)
print("Category:", ticket.category)
print("Priority:", ticket.priority)
print("Tags    :", ticket.tags)

# Convert it to a regular dictionary when needed
print("\nAs a dictionary:")
print(ticket.model_dump())

Summary : Customer reports being charged twice for their subscription in the current month and requests a refund ASAP.
Category: billing
Priority: high
Tags    : ['double-charged', 'subscription', 'refund-request', 'urgent']

As a dictionary:
{'summary': 'Customer reports being charged twice for their subscription in the current month and requests a refund ASAP.', 'category': 'billing', 'priority': 'high', 'tags': ['double-charged', 'subscription', 'refund-request', 'urgent']}


<br>

## Nested Data

Schemas can contain other Pydantic models. This is useful when the output naturally has sections, such as customer details inside a ticket.

In [5]:
class Customer(BaseModel):
    name: str
    email: str


class TicketWithCustomer(BaseModel):
    customer: Customer
    summary: str
    category: Literal["billing", "technical", "account", "other"]


nested_response = client.responses.parse(
    model="gpt-5.4-nano",
    input="Ana (ana@example.com) says she cannot reset her password.",
    text_format=TicketWithCustomer,
)

nested_ticket = nested_response.output_parsed
print(nested_ticket.customer.email)
print(nested_ticket.summary)

ana@example.com
Ana cannot reset her password.


<br>

## Important Notes

When using client.responses.parse() with a Pydantic model, OpenAI Structured Outputs enforces the corresponding supported JSON Schema, so the output conforms to the schema when generation completes successfully.

However, it does not guarantee that:
- The data is correct. A field such as age: 30 can be schema-valid but factually wrong.
- Every request succeeds. The model can refuse, or the response can be incomplete.


<br>

## Structured Outputs vs. Tool Calling

- Use **Structured Outputs** when you want the model's response in a predictable format, such as extraction, classification, or form filling.
- Use **tool calling** when the model needs your application to run a function or interact with an external system.
- They can be combined: tool calls can gather information, and Structured Outputs can format the final result.


<br>
<hr>
<hr>
<br>

## Practice: Structured Outputs


Instructions:
- https://gist.github.com/luisjunco/d6e4b11eeb5777557ecc859881a96296

Time: ~ 20 minutes.

<br>
<hr>
<hr>
<br>

